# Lexical vs semantic: consistency check, examples and user groups

This notebook compares the existing TF-IDF CBF (`src/content_based.py`) with the existing embedding CBF
(`src/semantic_embedding.py` and `src/semantic_based.py`) without changing either. Both are fitted on
`data/interactions_train.csv` only and evaluated on validation with the shared protocol in
`src/shared_eval.py`. The test split is not read.

Sections:
1. Shared protocol and evaluable users
2. Fit both models
3. Main metrics on the same users
4. Top-10 rule check
5. Consistency check TF-IDF vs embedding
6. Three example users (seed 42)
7. User activity groups (RQ2, separate analysis)

In [1]:
import json
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.shared_eval import build_protocol, summarize, check_top_k
from src.content_based import TfidfContentRecommender, load_item_text
from src.embedding_cache import ensure_item_embeddings
from src.semantic_based import SemanticEmbeddingRecommender
from src import semantic_embedding as se
from src import user_groups

import sklearn, scipy, torch, sentence_transformers
K = 10
SEED = 42
display(pd.Series({
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scipy': scipy.__version__,
    'scikit-learn': sklearn.__version__,
    'torch': torch.__version__,
    'sentence-transformers': sentence_transformers.__version__,
    'cuda available': torch.cuda.is_available(),
}, name='version').to_frame())

,version
python,3.11.4
numpy,1.24.3
pandas,1.5.3
scipy,1.10.1
scikit-learn,1.3.0
torch,2.15.0+cpu
sentence-transformers,6.2.0
cuda available,False


## 1. Shared protocol

`build_protocol` returns train, validation, the warm candidate set, the items each user saw in train,
and the relevant validation items per evaluable user (rating >= 4, warm candidate, user present in train).
It has the same name and arguments as the function on `feature/cp3-protocol-evaluate`.

In [2]:
data = build_protocol(str(ROOT / 'data') + '/')
train, val, cand_set, seen, relevant = data
positive_users = set(train.loc[train['rating'] >= 4, 'user_id'])
official = set(pd.read_csv(ROOT / 'data' / 'evaluable_users_validation_warm.csv')['user_id'])

display(pd.Series({
    'train interactions': len(train),
    'validation interactions': len(val),
    'warm candidates': len(cand_set),
    'evaluable users (used in all evaluations)': len(relevant),
    'evaluable users with a positive train item': len(set(relevant) & positive_users),
    'users in evaluable_users_validation_warm.csv': len(official),
    'official list == evaluable users with a positive train item': official == (set(relevant) & positive_users),
}, name='value').to_frame())

,value
train interactions,427957
validation interactions,41284
warm candidates,22753
evaluable users (used in all evaluations),11850
evaluable users with a positive train item,11659
users in evaluable_users_validation_warm.csv,11659
official list == evaluable users with a positive train item,True


All evaluations below use the 11,850 evaluable users, as in the stored TF-IDF and embedding results.
The official file lists 11,659 users: the 191 users without any train rating >= 4 are not in it.
This difference is reported to the group and not changed here.

## 2. Fit both models

TF-IDF uses the stored baseline parameters. The embedding model loads the item embeddings created by
`src/semantic_embedding.py` (all-MiniLM-L6-v2, batch size 32, CPU). The cache in `outputs/embeddings/`
is reused when present; `outputs/logs/embedding_cache_run.json` records the encoding run.

In [3]:
item_text = load_item_text(ROOT / 'data' / 'items.csv', cand_set)
tfidf = TfidfContentRecommender(
    stop_words='english', min_df=2, max_df=0.8, sublinear_tf=True, min_rating=4,
).fit(train, cand_set, item_text)

cache_info = ensure_item_embeddings()
embedding = SemanticEmbeddingRecommender(min_rating=4).fit(train, cand_set)
models = {'TF-IDF': tfidf, 'Embedding': embedding}

cache_log_path = ROOT / 'outputs' / 'logs' / 'embedding_cache_run.json'
cache_log = json.loads(cache_log_path.read_text(encoding='utf-8')) if cache_log_path.exists() else {}
display(pd.Series({
    'TF-IDF vocabulary size': len(tfidf.vectorizer.vocabulary_),
    'embedding matrix shape': tuple(embedding.item_matrix.shape),
    'embedding loaded from cache in this run': cache_info['from_cache'],
    'embedding model': cache_log.get('model_name', se.MODEL_NAME),
    'max_seq_length': cache_log.get('max_seq_length'),
    'texts longer than max_seq_length (truncated)': cache_log.get('truncated_texts'),
    'truncated tokens': cache_log.get('truncated_tokens'),
    'encoding time (s)': cache_log.get('encode_seconds'),
    'encoding device': cache_log.get('device', {}).get('encoder_device'),
    'CUDA GPU available': cache_log.get('device', {}).get('cuda_gpu_available'),
}, name='value').to_frame())

,value
TF-IDF vocabulary size,26550
embedding matrix shape,"(22753, 384)"
embedding loaded from cache in this run,True
embedding model,sentence-transformers/all-MiniLM-L6-v2
max_seq_length,256
texts longer than max_seq_length (truncated),8276
truncated tokens,1772380
encoding time (s),541.4
encoding device,cpu
CUDA GPU available,None


## 3. Main metrics on the same users

Per-user metrics come from `evaluate_per_user` (same computation as the TF-IDF notebook). The same call
also writes the RQ2 group breakdown used in section 7. The last two columns compare with the stored files.

In [4]:
group_result, group_metrics, per_user, groups = user_groups.run(models=models, data=data, save=True)

stored = {
    'TF-IDF': json.loads((ROOT / 'outputs/metrics/tfidf_cbf_val.json').read_text(encoding='utf-8')),
    'Embedding': json.loads((ROOT / 'outputs/metrics/semantic_cbf_val.json').read_text(encoding='utf-8')),
}
rows = []
for name, df in per_user.items():
    m = summarize(df, K)
    rows.append({'model': name, **m,
                 'stored NDCG@10': stored[name]['NDCG@10'],
                 'max abs diff vs stored': max(abs(m[k] - stored[name][k])
                                               for k in ['Recall@10', 'NDCG@10', 'HitRate@10'])})
main_table = pd.DataFrame(rows).set_index('model')
display(main_table.style.format({**{c: '{:.6f}' for c in ['Recall@10', 'NDCG@10', 'HitRate@10', 'stored NDCG@10']},
                                 'max abs diff vs stored': '{:.1e}'}))

,Recall@10,NDCG@10,HitRate@10,n_users,stored NDCG@10,max abs diff vs stored
model,,,,,,
TF-IDF,0.012722,0.007452,0.021772,11850,0.007452,0.0e+00
Embedding,0.008860,0.005361,0.015696,11850,0.005361,1.7e-18


For information only: the same per-user results restricted to the 11,659 users of the official file.

In [5]:
info_rows = []
for name, df in per_user.items():
    info_rows.append({'model': name, **summarize(df[df['user_id'].isin(official)], K)})
display(pd.DataFrame(info_rows).set_index('model').style.format(
    {c: '{:.6f}' for c in ['Recall@10', 'NDCG@10', 'HitRate@10']}))

,Recall@10,NDCG@10,HitRate@10,n_users
model,,,,
TF-IDF,0.012612,0.007353,0.021100,11659
Embedding,0.008686,0.005228,0.014924,11659


## 4. Top-10 rule check

For every evaluable user: exactly 10 unique items, none seen in train, all from the warm candidate set.

In [6]:
recs = {name: {u: m.recommend(u, seen, k=K) for u in relevant} for name, m in models.items()}
display(pd.DataFrame({name: check_top_k(r, cand_set, seen, K) for name, r in recs.items()}))

,TF-IDF,Embedding
wrong_length,0,0
duplicate_items,0,0
seen_items,0,0
non_candidate_items,0,0


## 5. Consistency check TF-IDF vs embedding

Each row compares one part of the pipeline. Checks are computed from the fitted objects where possible.
Nothing in either model is changed.

In [7]:
# Item text: TF-IDF loader vs embedding loader.
emb_text = se.load_item_text(se.DEFAULT_ITEMS_PATH, cand_set)
same_ids = set(item_text) == set(emb_text)
raw_equal = sum(item_text[i] == emb_text[i] for i in item_text)
strip_equal = sum(str(item_text[i]).strip() == emb_text[i] for i in item_text)

# Profiles: same positive train items per user.
same_profile_users = set(tfidf.user_items) == set(embedding.user_items)
tf_ids = np.array(sorted(tfidf.item_index, key=tfidf.item_index.get))
em_ids = np.asarray(embedding.item_ids)
same_profile_items = same_profile_users and all(
    set(tf_ids[tfidf.user_items[u]]) == set(em_ids[embedding.user_items[u]]) for u in tfidf.user_items)

# Cosine: unit-norm rows, and score equals cosine for a sample profile.
tf_norms = np.sqrt(tfidf.cand_matrix.multiply(tfidf.cand_matrix).sum(axis=1)).A.ravel()
em_norms = np.linalg.norm(embedding.cand_matrix, axis=1)
tf_zero = tf_norms < 1e-12
zero_items = set(np.array(tfidf.candidate_items)[tf_zero])
zero_in_recs = sum(i in zero_items for r in recs['TF-IDF'].values() for i in r)
u0 = sorted(u for u in relevant if u in tfidf.user_items)[0]
p_tf, p_em = tfidf.user_profile(u0), embedding.user_profile(u0)
cos_tf = (tfidf.cand_matrix @ p_tf) / np.linalg.norm(p_tf)
rank_tf_raw = np.argsort(-(tfidf.cand_matrix @ p_tf), kind='stable')[:K]
rank_tf_cos = np.argsort(-cos_tf, kind='stable')[:K]

# Fallback: same popularity list and identical recs for users without a profile.
no_profile = [u for u in relevant if tfidf.user_profile(u) is None]
no_profile_em = [u for u in relevant if embedding.user_profile(u) is None]
same_fallback_recs = all(recs['TF-IDF'][u] == recs['Embedding'][u] for u in no_profile)

viol = {name: sum(check_top_k(r, cand_set, seen, K).values()) for name, r in recs.items()}
emb_m = main_table.loc['Embedding']

def status(ok):
    return 'same' if ok else 'different'

consistency = pd.DataFrame([
    ('Item text', 'content_text_basic, items.csv', 'content_text_basic, items.csv',
     status(same_ids and strip_equal == len(item_text)),
     f'{len(item_text):,} items; identical raw text {raw_equal:,}, identical after strip {strip_equal:,}. '
     'Embedding loader also uses utf-8-sig, drop_duplicates and strip; no effect on this data.'),
    ('Candidate set', f'{len(tfidf.candidate_items):,} warm items', f'{len(embedding.candidate_items):,} warm items',
     status(tfidf.candidate_items == embedding.candidate_items),
     'Same sorted candidate list. Embedding requires every candidate to have a vector; TF-IDF uses a zero row.'),
    ('User profile items', 'train rating >= 4', 'train rating >= 4',
     status(same_profile_items),
     f'{len(tfidf.user_items):,} users with a profile in both; same items per user.'),
    ('Profile vector', 'mean, not normalized', 'mean, then L2-normalized',
     'different (same ranking)',
     'Scaling the profile does not change the order of cosine scores; '
     f'top-10 order identical for a sample user: {bool((rank_tf_raw == rank_tf_cos).all())}.'),
    ('Scoring', 'dot product of unit rows', 'dot product of unit rows and unit profile',
     status(np.allclose(tf_norms[~tf_zero], 1) and np.allclose(em_norms, 1)),
     f'Both rank by cosine. Embedding rows all unit norm. TF-IDF: {int((~tf_zero).sum()):,} unit rows and '
     f'{int(tf_zero.sum())} all-zero row(s) (every term removed by stop words/min_df/max_df); '
     f'such items score 0 and appear {zero_in_recs} times in the TF-IDF top-10 lists.'),
    ('Train items removed', 'score set to -inf', 'score set to -inf', status(viol['TF-IDF'] == viol['Embedding'] == 0),
     'No recommended item was seen in train (section 4).'),
    ('Top-10', '10 unique items', '10 unique items', status(viol['TF-IDF'] == viol['Embedding'] == 0),
     'Positive scores first, then popularity fill.'),
    ('Popularity fallback', 'train rating >= 4 counts', 'train rating >= 4 counts',
     status(tfidf.ranked_items == embedding.ranked_items and set(no_profile) == set(no_profile_em) and same_fallback_recs),
     f'Same ranked list; {len(no_profile)} users without a profile get identical recommendations.'),
    ('Evaluation function', 'evaluate_at_k in TF-IDF notebook', 'evaluate_semantic_at_k in semantic notebook',
     'same formula',
     'Both are separate copies of the same Recall/NDCG/HitRate formula. Here both models use src/shared_eval.py. '
     f'Embedding NDCG@10 here {emb_m["NDCG@10"]:.6f} vs stored {stored["Embedding"]["NDCG@10"]:.6f} '
     f'(max abs diff {emb_m["max abs diff vs stored"]:.1e}, floating-point summation order).'),
    ('Evaluable users', f'{stored["TF-IDF"]["n_eval_users"]:,}', f'{stored["Embedding"]["n_eval_users"]:,}',
     status(stored['TF-IDF']['n_eval_users'] == stored['Embedding']['n_eval_users'] == len(relevant)),
     'Both use load_data + relevant_items_val (min_rating 4).'),
], columns=['aspect', 'TF-IDF (Tiffani)', 'Embedding (Lenni)', 'status', 'detail']).set_index('aspect')
with pd.option_context('display.max_colwidth', None):
    display(consistency)

,TF-IDF (Tiffani),Embedding (Lenni),status,detail
aspect,,,,
Item text,"content_text_basic, items.csv","content_text_basic, items.csv",same,"22,753 items; identical raw text 22,753, identical after strip 22,753. Embedding loader also uses utf-8-sig, drop_duplicates and strip; no effect on this data."
Candidate set,"22,753 warm items","22,753 warm items",same,Same sorted candidate list. Embedding requires every candidate to have a vector; TF-IDF uses a zero row.
User profile items,train rating >= 4,train rating >= 4,same,"54,036 users with a profile in both; same items per user."
Profile vector,"mean, not normalized","mean, then L2-normalized",different (same ranking),Scaling the profile does not change the order of cosine scores; top-10 order identical for a sample user: True.
Scoring,dot product of unit rows,dot product of unit rows and unit profile,same,"Both rank by cosine. Embedding rows all unit norm. TF-IDF: 22,752 unit rows and 1 all-zero row(s) (every term removed by stop words/min_df/max_df); such items score 0 and appear 0 times in the TF-IDF top-10 lists."
Train items removed,score set to -inf,score set to -inf,same,No recommended item was seen in train (section 4).
Top-10,10 unique items,10 unique items,same,"Positive scores first, then popularity fill."
Popularity fallback,train rating >= 4 counts,train rating >= 4 counts,same,Same ranked list; 191 users without a profile get identical recommendations.
Evaluation function,evaluate_at_k in TF-IDF notebook,evaluate_semantic_at_k in semantic notebook,same formula,"Both are separate copies of the same Recall/NDCG/HitRate formula. Here both models use src/shared_eval.py. Embedding NDCG@10 here 0.005361 vs stored 0.005361 (max abs diff 1.7e-18, floating-point summation order)."


## 6. Three example users

The same 3 users as `outputs/samples/tfidf_cbf_examples.md`: picked with `np.random.default_rng(42)`
from evaluable users that have a TF-IDF profile.

In [8]:
titles = pd.read_csv(ROOT / 'data' / 'items.csv', dtype={'item_id': str},
                     usecols=['item_id', 'title']).set_index('item_id')['title']
rng = np.random.default_rng(SEED)
pool = sorted(u for u in relevant if u in tfidf.user_items)
sample_users = rng.choice(pool, size=3, replace=False).tolist()

def short(item, n=80):
    t = titles.get(item)
    t = '(no title)' if pd.isna(t) else str(t)
    return t if len(t) <= n else t[:n - 3] + '...'

for user_id in sample_users:
    liked = train[(train['user_id'] == user_id) & (train['rating'] >= 4)]
    display(Markdown(f'### User `{user_id}`\n{len(liked)} items liked in train; '
                     f'{len(relevant[user_id])} relevant items in validation.'))
    display(pd.DataFrame({'item_id': liked['item_id'].to_numpy(),
                          'title': [short(i) for i in liked['item_id']],
                          'rating': liked['rating'].astype(int).to_numpy()}).style.hide(axis='index'))
    side = pd.DataFrame({
        'rank': range(1, 6),
        'TF-IDF top-5': [short(i, 60) for i in recs['TF-IDF'][user_id][:5]],
        'TF-IDF hit': [i in relevant[user_id] for i in recs['TF-IDF'][user_id][:5]],
        'Embedding top-5': [short(i, 60) for i in recs['Embedding'][user_id][:5]],
        'Embedding hit': [i in relevant[user_id] for i in recs['Embedding'][user_id][:5]],
    })
    display(side.style.hide(axis='index'))
    overlap = len(set(recs['TF-IDF'][user_id]) & set(recs['Embedding'][user_id]))
    display(Markdown(f'Overlap between the two top-10 lists: {overlap} of 10 items.'))

### User `AGO3MQHEHK3XE2VJWGXYARGOVDPQ`
7 items liked in train; 3 relevant items in validation.

item_id,title,rating
B0016HDP58,Gibraltar SC-CMBAC Medium Cymbal Boom Attachment Clamp,5
B014P7CDDU,Musiclily Pickguard Strat Back Plate Tremolo Cavity Cover Vintage Style Backp...,5
B07DVX841V,Musiclily Pro 6 inline Guitar Locking Tuners String Tuning Pegs Keys Machines...,5
B0B92LPGD2,Fender American Series String Guides,5
B083KJPCVV,Ubblove Drumsticks 5A 2 pairs with ANTI-SLIP Handles for Drum Light Durable W...,5
B0B8Z6192W,Fender 5-way Selector Switch,5
B015IJIO5U,VOX AP2BS amPlug 2 Guitar/Bass Headphone Amplifier Black,5


rank,TF-IDF top-5,TF-IDF hit,Embedding top-5,Embedding hit
1,Musiclily 6 Hole Strat Back Plate Tremolo Cavity Cover Ba...,False,GETMusic 3MM Electric Guitar Bass Pickguard Screws Pick G...,False
2,Musiclily Guitar Flat Wavy Strat String Guides String Tre...,False,Well Strung Guitar Parts Accessory Kit for Fender Stratoc...,False
3,Musiclily 6-in-line Guitar Locking Tuners Machine Head Tu...,False,Musiclily Pro 11 Hole Loaded Guitar Strat Pickguard HSS w...,False
4,Musiclily 13 Hole P Bass Pickguard Precision Bass Scratch...,False,Musiclily Pro 11 Hole Loaded Strat Pickguard with Alnico ...,False
5,Musiclily Pro 10 Hole Jazz Bass Pickguard J Bass Scratch ...,False,Musiclily Metal Vintage Style String Guides for Strat or ...,False


Overlap between the two top-10 lists: 1 of 10 items.

### User `AEFDMLY3LCEB4U7Y5IFOUFNQ4ZTQ`
5 items liked in train; 2 relevant items in validation.

item_id,title,rating
B001GH4H3W,[OLD MODEL] Vox AmPlug Bass Guitar Headphone Amp,5
B007IE5TQG,"Hohner Special 20 Harmonica 3 piece Pro Pack keys of G,C,A",5
B09JXTHJKK,"Mendini By Cecilio Eb Alto Saxophone - Case, Tuner, Mouthpiece, 10 Reeds, Poc...",5
B07WWZKZY9,Amazon Basics Sax Stand with Flute/Clarinet Peg,5
B086W93GHN,"Eno Solid Big Clip-on Professional Wind Instruments Alto Saxophone Tuner, Col...",4


... (2 outputs left out)

### User `AH4PQ6UJM6FVZXFOC7AFZY2G4B4Q`
21 items liked in train; 4 relevant items in validation.

item_id,title,rating
B07CZMZRWN,Soulo SM8525 Trumpet Harmon Style Wah Wah Mute,5
B01FRN3N1C,Gammon Percussion Full Size Complete Adult 5 Piece Drum Set with Cymbals Stan...,5
B01M18LT95,Gearlux Attachable Music Stand Arm - Small,5
B077J6ZGD5,Hola! Music HM-MTH Microphone Music Stand iPad Tablet Smartphone Holder Mount...,5
B00SSVYYXW,Clarinet Oboe Soprano Saxophone Electronic Torch Sax Gig Bag Hot,5
B0BMV8CDXJ,Otto Link AOL OLMT6 Accordion Accessory (OLM-404-6),5
B09VP8Y3MV,AKAI Professional EWI Solo - Electronic Wind Instrument With Built-in Speaker...,5
B00KDE3U5K,"AxeRig LIFETIME WARRANTY Clip-On Chromatic Guitar Tuner for Acoustic, Bass, 6...",5
B004LLV04E,"PreSonus AudioBox Studio with Headphones, Microphone, Mic Cable, USB Cable, a...",5
B0C34Z8KSM,"Hohner 1896BX Marine Band Harmonic, Key of C Sharp",5


rank,TF-IDF top-5,TF-IDF hit,Embedding top-5,Embedding hit
1,"East top Harmonica, Blues Harmonica Key of E, 10 Holes 00...",False,Seydel Harmonica (Blues SESSION Steel G),False
2,East top Diatonic Harmonica Key of D 10 Holes 20 Tones 00...,False,Harmonica Professional Harp Armonica Key of C 24 Hole Pol...,False
3,East top Diatonic Harmonica Key of C 10 Holes 20 Tones 00...,False,"Hohner Special 20 Harmonica, Key of G",False
4,"East top Harmonica Set of 7, 10 Holes Blues Harp Mouth Or...",False,"Harmonica, YINGXIANG Harmonicas, Standard Diatonic 10 Hol...",False
5,"East top Diatonic Harmonica Key of Eb, 10 Holes 20 Tones ...",False,"Hohner Harmonica, Chrome (590BX-C)",False


Overlap between the two top-10 lists: 2 of 10 items.

## 7. User activity groups (RQ2)

Users are labelled by their number of train interactions using `configs/user_groups.json`
(rendah <= 5, sedang 6-8, tinggi >= 9). The groups only break down the metrics computed above; they do not
filter the main evaluation. Results are saved to `outputs/metrics/user_groups_val.json`.

In [9]:
bounds = pd.DataFrame(group_result['group_bounds']).set_index('label').astype('Int64').astype(str).replace('<NA>', '-')
bounds.columns = ['min train interactions', 'max train interactions']
counts = pd.DataFrame(group_result['evaluable_users_per_group']).T
counts.insert(0, 'train users', pd.Series(group_result['n_train_users_per_group']))
display(bounds.join(counts))

,min train interactions,max train interactions,train users,n_evaluable_users,n_without_positive_train_item
label,,,,,
rendah,-,5,22117,6295,190
sedang,6,8,18197,2365,1
tinggi,9,-,14169,3190,0


In [10]:
wide = group_metrics.pivot(index='group', columns='model', values=['NDCG@10', 'Recall@10', 'HitRate@10'])
wide = wide.reindex(index=['rendah', 'sedang', 'tinggi', 'all'],
                    columns=pd.MultiIndex.from_product([['NDCG@10', 'Recall@10', 'HitRate@10'],
                                                        ['TF-IDF', 'Embedding']]))
wide.insert(0, ('n_users', ''), group_metrics.drop_duplicates('group').set_index('group')['n_users'])
wide[('NDCG@10', 'Embedding - TF-IDF')] = wide[('NDCG@10', 'Embedding')] - wide[('NDCG@10', 'TF-IDF')]
display(wide.style.format('{:.6f}', subset=[c for c in wide.columns if c[0] != 'n_users']))